In [ ]:
# Load data, extract exchange rates, and normalize them
import pandas as pd
import numpy as np
from scipy.signal import argrelextrema
import altair as alt
# alt.data_transformers.enable('default')

# Load the CSV file into a pandas DataFrame
file_path = 'priceIndex_and_exchangeRate_series_2025_SP.csv'  # Replace with your CSV file path
data = pd.read_csv(file_path, index_col=0)

african_countries = [
    'DZA', 'AGO', 'BEN', 'BWA', 'BFA', 'BDI', 'CMR', 'CPV', 'CAF', 'TCD', 'COM', 'COG', 'CIV', 
    'COD', 'DJI', 'EGY', 'GNQ', 'ERI', 'SWZ', 'ETH', 'GAB', 'GMB', 'GHA', 'GIN', 'GNB', 'KEN', 
    'LBY', 'LSO', 'LBR', 'MDG', 'MWI', 'MLI', 'MRT', 'MUS', 'MAR', 'MOZ', 'NAM', 'NER', 'NGA', 
    'RWA', 'STP', 'SEN', 'SYC', 'SLE', 'SOM', 'ZAF', 'SSD', 'SDN', 'TZA', 'TGO', 'TUN', 'UGA', 
    'ZMB', 'ZWE'
]

def build_hybrid_index_and_ma_df(
    start="1990-01-01",
    end="2023-01-01",
    ma_window=5,
    countries=african_countries,
    df=data
):
    """
    Build a hybrid real exchange rate index and its moving average for specified countries.
    
    This function creates a real exchange rate index by dividing the local price index
    by the nominal exchange rate, normalized to 100 at the first valid observation. A falling 
    index indicates currency depreciation/devaluation.
    
    Parameters
    ----------
    start : str, default "1990-01-01"
        Start date for the analysis period in YYYY-MM-DD format.
    end : str, default "2023-01-01"
        End date for the analysis period in YYYY-MM-DD format.
    ma_window : int, default 5
        Window size in years for the moving average baseline calculation.
    countries : list, default african_countries
        List of 3-letter ISO country codes to process.
    df : pd.DataFrame, default data
        Input DataFrame with exchange rate and price data. Expected columns:
        - {COUNTRY}PANUSATLS: Nominal exchange rate (LCU per USD)
        - {COUNTRY}NECONPRVTXN: Consumer price index
        
    Returns
    -------
    pd.DataFrame
        Tidy DataFrame with columns:
        - country: 3-letter ISO country code
        - year: Year as integer
        - fx_index: Real exchange rate index (normalized to 100)
        - baseline: Moving average of fx_index over ma_window years
        
    Notes
    -----
    - Countries with missing data columns are skipped with a warning
    - The function requires ma_window years of data before start date for MA calculation
    - All rows with NaN values are dropped from the final result
    - Index normalization ensures cross-country comparability
    
    Examples
    --------
    >>> # Get data for 2000-2020 with 3-year moving average
    >>> df = build_hybrid_index_and_ma_df(
    ...     start="2000-01-01", 
    ...     end="2020-01-01", 
    ...     ma_window=3
    ... )
    """

    start_date = pd.to_datetime(start)
    sd_minus_window = start_date - pd.DateOffset(years=ma_window-1) 
    df.index = pd.to_datetime(df.index)
    df = df[(df.index >= sd_minus_window) & (df.index <= end)]
    
    records = []

    for country in countries:
        panusatls_col = f"{country}PANUSATLS"
        price_level_col = f"{country}NECONPRVTXN"

        if panusatls_col not in df.columns or price_level_col not in df.columns:
            print(f"Missing columns for {country}")
            continue

        series = df[price_level_col] / df[panusatls_col]
        first_valid = series.first_valid_index()
        if first_valid is None:
            print(f"No data for {country}")
            continue

        # normalize to 100 at first valid point
        series_norm = series / series.loc[first_valid] * 100
        series_ma = series_norm.rolling(window=ma_window, min_periods=ma_window).mean()

        for date, index_val, ma_val in zip(series_norm.index, series_norm.values, series_ma.values):
            if date >= start_date and not pd.isna(index_val):
                records.append({
                    "country": country,
                    "year": date.year,
                    "fx_index": float(index_val),
                    "baseline": float(ma_val) if not pd.isna(ma_val) else np.nan
                })
                
    tidy_df = pd.DataFrame(records).sort_values(["country", "year"]).reset_index(drop=True)
    return tidy_df.dropna()

def identify_events(
    df, 
    trigger_threshold = 0.1,
    end_threshold = 0.1,
    countries = african_countries
):
    """
    Identify currency devaluation events using peak detection and threshold analysis.
    
    This function detects significant currency devaluation episodes by:
    1. Finding local troughs in the real exchange rate index
    2. Identifying events where troughs exceed trigger_threshold below baseline
    3. Determining event end when index returns to end_threshold within baseline
    4. Filtering overlapping events to select the most severe ones
    
    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame from build_hybrid_index_and_ma_df with columns:
        - country: 3-letter ISO country code
        - year: Year as integer  
        - fx_index: Real exchange rate index
        - baseline: Moving average baseline
    trigger_threshold : float, default 0.1
        Minimum deviation below baseline (as fraction) to trigger an event.
        E.g., 0.1 = 10% below baseline.
    end_threshold : float, default 0.1
        Maximum deviation from baseline (as fraction) to end an event.
        E.g., 0.0 = return to baseline level.
    countries : list, default african_countries
        List of 3-letter ISO country codes to analyze.
        
    Returns
    -------
    pd.DataFrame
        DataFrame of identified devaluation events with columns:
        - Country: 3-letter ISO country code
        - Start_Date: Year when event began (first point below trigger)
        - End_Date: Year when event ended (return back above end threshold) or last data year
        - Peak_Date: Year of maximum devaluation during event
        - Duration: Event length in years
        - Peak_Devaluation: Maximum percentage deviation from baseline during event
        - Devaluation_at_End: Percentage deviation from baseline at event end
        - Drawdown_Area: Cumulative percentage-point loss over event duration
        
    Notes
    -----
    Algorithm Details:
    - Uses scipy.signal.argrelextrema to find local minima (troughs)
    - Requires "clean" fall: no intermediate peaks above trigger during run-up
    - For overlapping events, selects those with greatest drawdown area first,
      then lowest peak devaluation as tiebreaker
    - Ongoing events (no return to threshold) use end of sample as event end
    - Drawdown area calculated as cumulative deviation from baseline over time
    
    Event Quality Filters:
    - Skips events with intermediate peaks during fall phase
    - Handles incomplete events that don't return to baseline
    - Prevents double-counting of overlapping episodes
    
    Examples
    --------
    >>> # Detect major devaluation events (>20% trigger, return to baseline)
    >>> events = identify_events(
    ...     df, 
    ...     trigger_threshold=0.20,
    ...     end_threshold=0.0
    ... )
    
    >>> # Detect all significant events (>10% trigger, partial recovery to 5%)
    >>> events = identify_events(
    ...     df,
    ...     trigger_threshold=0.10, 
    ...     end_threshold=0.05
    ... )
    """

    # Add in_event column to dataframe
    if 'in_event' not in df.columns:
        df['in_event'] = 0
    # If annual_loss column doesn't exist, create it
    if 'annual_loss' not in df.columns:
        df['annual_loss'] = 0.0
    results = []

    for country in countries:
        country_data = df[df.country == country]

        # Check if country has any data in the filtered dataframe
        if len(country_data) == 0:
            continue  # Skip countries with no data
        
        year         = country_data["year"].values
        fx_index     = country_data["fx_index"].values
        baseline     = country_data["baseline"].values

        troughs_new_indices = argrelextrema(fx_index, np.less)[0]
        if len(fx_index) >= 2 and fx_index[-1] < fx_index[-2]:
            troughs_new_indices = np.append(troughs_new_indices, len(fx_index) - 1)

        # Store events for this country to filter overlaps
        country_events = []

        for p_new_idx in np.arange(len(year)):
            trigger_level = baseline[p_new_idx] * (1. - trigger_threshold)  # Use trigger threshold
            end_level     = baseline[p_new_idx] * (1. - end_threshold)  # This changes with end_threshold!

            possible_event_troughs = troughs_new_indices[troughs_new_indices > p_new_idx]
            if possible_event_troughs.size == 0:
                continue

            for t_new_idx in possible_event_troughs:

                if fx_index[t_new_idx] <= trigger_level:
                    end_date = None
                    event_duration = None
                    max_drop = None
                    max_devaluation = None
                    drawdown_area = None
                    
                    intermediate_troughs = 0
                    for i in range(p_new_idx+1, t_new_idx):
                        if fx_index[i] > trigger_level:  # Check against trigger level
                            intermediate_troughs += 1
                    if intermediate_troughs > 0:
                        continue # skip this event if there are intermediate troughs, as it is not a clean rise and fall

                    for idx in range(t_new_idx+1, len(fx_index)):
                        if fx_index[idx] >= end_level:
                            # time measures
                            end_date = year[idx]
                            event_duration = year[idx] - year[p_new_idx]

                            # max amplitude
                            max_drop = t_new_idx
                            for i in troughs_new_indices[(troughs_new_indices>t_new_idx) & (troughs_new_indices<idx)]:
                                if fx_index[i] < fx_index[max_drop]:
                                    max_drop = i
                            max_devaluation = (fx_index[max_drop] - baseline[p_new_idx]) / baseline[p_new_idx] * 100

                            # total dollar loss in precent of CFADS
                            # Replace fx_index values at start and end with baseline in a copy
                            baseline_to_baseline        = np.array(fx_index, copy = True)
                            # baseline_to_baseline[p_new_idx] = np.maximum(baseline_to_baseline[p_new_idx], baseline[p_new_idx])
                            # baseline_to_baseline[idx]   = np.maximum(baseline_to_baseline[idx],  baseline[p_new_idx])

                            integrand = np.minimum(baseline_to_baseline/baseline[p_new_idx] - 1, 0.0) * 100 # percentage deviation from baseline at event start date
                            drawdown_area = np.sum(integrand[p_new_idx:idx+1]) #sum of integrand over event period
                            break

                    else: # if return condition not met, repeat with end period
                        idx = len(fx_index) -1
                        end_date = year[idx]
                        event_duration = year[idx] - year[p_new_idx]
                        max_drop = t_new_idx
                        for i in troughs_new_indices[(troughs_new_indices>t_new_idx)]:
                            if fx_index[i] > fx_index[max_drop]:
                                max_drop = i
                        max_devaluation = (fx_index[max_drop] - baseline[p_new_idx]) / baseline[p_new_idx] * 100
                        baseline_to_baseline        = np.array(fx_index, copy = True)
                        # baseline_to_baseline[p_new_idx] = np.maximum(baseline_to_baseline[p_new_idx], baseline[p_new_idx])
                        # baseline_to_baseline[idx]   = np.maximum(baseline_to_baseline[idx],  baseline[p_new_idx])

                        integrand = np.minimum(baseline_to_baseline/baseline[p_new_idx] - 1, 0.0) * 100 # percentage deviation from baseline at event start date
                        drawdown_area = np.sum(integrand[p_new_idx:idx+1]) #sum of integrand over event period

                    country_events.append({
                        "Country": country, 
                        "Start_Date": year[p_new_idx],
                        "End_Date": end_date,
                        "Peak_Date": year[max_drop],
                        "Duration": event_duration,
                        "Peak_Devaluation": max_devaluation,
                        "Devaluation_at_End": (fx_index[idx] - baseline[p_new_idx]) / baseline[p_new_idx] * 100,
                        "Drawdown_Area": drawdown_area
                    })

        # Filter overlapping events for this country
        selected = []
        if country_events:
            # Sort by Drawdown_Area ascending, then by Peak_Devaluation ascending
            country_events.sort(key=lambda x: (x['Drawdown_Area'], x['Peak_Devaluation'])) # , reverse=True
            
            
            used_intervals = []
            for event in country_events:
                start = event['Start_Date']
                end = event['End_Date']
                # Check for overlap
                overlap = False
                for s, e in used_intervals:
                    if not (end < s or start > e):
                        overlap = True
                        break
                if not overlap:
                    selected.append(event)
                    used_intervals.append((start, end))
            
            results.extend(selected)

            
        # Mark country-years as being in an event
        for event in selected:
            start = event['Start_Date']
            end = event['End_Date']
            event_years_mask = (
                (df['country'] == country) & 
                (df['year'] >= start) & 
                (df['year'] <= end)
            )
            # Mark rows as being in an event
            df.loc[event_years_mask, 'in_event'] = 1
            
            # Get baseline at event start date for reference
            start_baseline = df.loc[(df['country'] == country) & (df['year'] == start), 'baseline'].values[0]
            
            # Calculate annual loss for each year in the event
            for idx in df.loc[event_years_mask].index:
                current_fx = df.loc[idx, 'fx_index']
                loss = (current_fx - start_baseline) / start_baseline
                # Only record negative deviations (actual losses)
                df.loc[idx, 'annual_loss'] = min(loss, 0.0)

    return pd.DataFrame(results), df

def create_annual_event_data(events, hdf):
    """
    Create annual prevalence data by expanding multi-year events into individual country-years.
    
    This function transforms the event-level data into annual observations, where each
    country-year during an event gets its own record with the annual drawdown loss.
    
    Parameters
    ----------
    events : pd.DataFrame
        Event-level data from identify_events() with columns:
        - Country, Start_Date, End_Date, Duration, Drawdown_Area
    hdf : pd.DataFrame
        Country-year data from build_hybrid_index_and_ma_df() with columns:
        - country, year, fx_index, baseline
        
    Returns
    -------
    pd.DataFrame
        Annual event data with columns:
        - Country: 3-letter ISO country code
        - Year: Year as integer
        - In_Event: Binary indicator (1 if country-year is in devaluation event)
        - Annual_Loss: Annual drawdown loss for that year (negative percentage)
        - Event_Year: Year within the event (1, 2, 3, etc.)
        - Total_Event_Duration: Total duration of the event this year belongs to
    """
    
    # Create base dataset with all country-years
    annual_data = hdf[['country', 'year', 'fx_index', 'baseline']].copy()
    annual_data.rename(columns={'country': 'Country', 'year': 'Year'}, inplace=True)
    annual_data['In_Event'] = 0
    annual_data['Annual_Loss'] = 0.0
    annual_data['Event_Year'] = 0
    annual_data['Total_Event_Duration'] = 0
    
    if len(events) == 0:
        return annual_data
    
    # Process each event to create annual records
    for _, event in events.iterrows():
        country = event['Country']
        start_year = int(event['Start_Date'])
        end_year = int(event['End_Date'])
        duration = int(event['Duration'])
        total_drawdown = event['Drawdown_Area']  # Total cumulative loss over entire event
        
        # Get country data for this event period
        country_mask = (annual_data['Country'] == country) & \
                      (annual_data['Year'] >= start_year) & \
                      (annual_data['Year'] <= end_year)
        
        event_data = annual_data[country_mask].copy()
        
        if len(event_data) == 0:
            continue
            
        # Calculate annual losses during the event
        for i, (idx, row) in enumerate(event_data.iterrows()):
            year = row['Year']
            fx_index = row['fx_index']
            
            # Get baseline at start of event for consistent reference
            event_start_baseline = annual_data[
                (annual_data['Country'] == country) & 
                (annual_data['Year'] == start_year)
            ]['baseline'].iloc[0]
            
            # Annual loss as percentage deviation from event start baseline
            annual_loss = (fx_index - event_start_baseline) / event_start_baseline * 100
            # Ensure loss is negative (below baseline)
            annual_loss = min(annual_loss, 0.0)
            
            # Update the annual data
            annual_data.loc[idx, 'In_Event'] = 1
            annual_data.loc[idx, 'Annual_Loss'] = annual_loss
            annual_data.loc[idx, 'Event_Year'] = i + 1
            annual_data.loc[idx, 'Total_Event_Duration'] = duration
    
    return annual_data

def frequency_and_severity_pooled(events, hdf):
    """
    Calculate pooled portfolio frequency and severity metrics using annual prevalence approach.
    
    This function uses an annual prevalence model where frequency is the proportion of
    country-years experiencing losses, and severity is the distribution of annual losses.
    This approach is more appropriate for annual capital requirements.
    
    Parameters
    ----------
    events : pd.DataFrame
        Event-level data from identify_events()
    hdf : pd.DataFrame
        Country-year data from build_hybrid_index_and_ma_df()
        
    Returns
    -------
    dict
        Dictionary with annual prevalence metrics:
        - frequency: Proportion of country-years with losses (annual prevalence)
        - mean_severity: Mean annual loss when losses occur
        - std_severity: Standard deviation of annual losses
        - p95_severity: 95th percentile of annual losses
        - p99_severity: 99th percentile of annual losses  
        - expected_loss: Expected annual loss per country-year
    """
    
    # Create annual event data
    annual_data = create_annual_event_data(events, hdf)
    
    # Total country-year exposure
    total_country_years = len(annual_data)
    
    if total_country_years == 0:
        return {
            'frequency': 0.0,
            'mean_severity': 0.0,
            'std_severity': 0.0,
            'p95_severity': 0.0,
            'p99_severity': 0.0,
            'expected_loss': 0.0
        }
    
    # Annual prevalence frequency (proportion of country-years with losses)
    event_years = annual_data[annual_data['In_Event'] == 1]
    frequency = len(event_years) / total_country_years
    
    if len(event_years) == 0:
        return {
            'frequency': 0.0,
            'mean_severity': 0.0,
            'std_severity': 0.0,
            'p95_severity': 0.0,
            'p99_severity': 0.0,
            'expected_loss': 0.0
        }
    
    # Severity statistics for annual losses (when losses occur)
    annual_losses = event_years['Annual_Loss'].abs()  # Take absolute value for positive severity
    mean_severity = annual_losses.mean()
    std_severity = annual_losses.std()
    p95_severity = annual_losses.quantile(0.95)
    p99_severity = annual_losses.quantile(0.99)
    
    # Expected annual loss per country-year
    expected_loss = frequency * mean_severity
    
    return {
        'frequency': frequency,
        'mean_severity': mean_severity,
        'std_severity': std_severity,
        'p95_severity': p95_severity,
        'p99_severity': p99_severity,
        'expected_loss': expected_loss
    }

def frequency_and_severity_by_country(events, hdf):
    """
    Calculate country-level frequency and severity metrics using annual prevalence approach.
    
    This function calculates country-specific annual prevalence rates and severity
    distributions, providing a more granular view for portfolio management.
    
    Parameters
    ----------
    events : pd.DataFrame
        Event-level data from identify_events()
    hdf : pd.DataFrame
        Country-year data from build_hybrid_index_and_ma_df()
        
    Returns
    -------
    pd.DataFrame
        DataFrame with country-level annual prevalence metrics:
        - Country: 3-letter ISO country code
        - Total_Years: Total country-year exposure
        - Event_Years: Number of country-years with losses
        - Frequency: Annual prevalence rate (Event_Years / Total_Years)
        - Mean_Severity: Mean annual loss when losses occur
        - Expected_Loss: Expected annual loss (Frequency * Mean_Severity)
    """
    
    # Create annual event data
    annual_data = create_annual_event_data(events, hdf)
    
    # Calculate country-level exposure
    exposure = annual_data.groupby('Country').agg({
        'Year': 'count',
        'In_Event': 'sum',
        'Annual_Loss': lambda x: x[x != 0].abs().mean() if (x != 0).any() else 0.0
    }).reset_index()
    
    exposure.columns = ['Country', 'Total_Years', 'Event_Years', 'Mean_Severity']
    
    # Calculate frequency (annual prevalence)
    exposure['Frequency'] = exposure['Event_Years'] / exposure['Total_Years']
    
    # Calculate expected annual loss
    exposure['Expected_Loss'] = exposure['Frequency'] * exposure['Mean_Severity']
    
    # Fill NaN values
    exposure = exposure.fillna(0.0)
    
    return exposure[['Country', 'Total_Years', 'Event_Years', 'Frequency', 'Mean_Severity', 'Expected_Loss']]

def annuity_factor(n_years=15, r=0.05):
    """
    Calculate present value annuity factor for converting lump sum to annual payments.
    
    Parameters
    ----------
    n_years : int, default 15
        Number of years for the annuity
    r : float, default 0.05
        Annual discount rate
        
    Returns
    -------
    float
        Present value annuity factor
    """
    if r == 0:
        return n_years
    return (1 - (1 + r) ** (-n_years)) / r

def calculate_annual_prevalence_metrics(df_events):
    """
    Calculate streamlined annual prevalence metrics for insurance premium calculation.
    
    This function computes the core actuarial statistics needed for annual prevalence pricing:
    frequency (proportion of country-years in events) and severity (mean annual loss relative
    to event start baseline).
    
    Parameters
    ----------
    df_events : pd.DataFrame
        Augmented dataframe from identify_events() with columns:
        - country: 3-letter ISO country code
        - year: Year as integer
        - fx_index: Real exchange rate index
        - baseline: Moving average baseline
        - in_event: Binary indicator (1 if in event, 0 otherwise)
        
    Returns
    -------
    dict
        Dictionary with essential annual prevalence metrics:
        - frequency: Annual prevalence rate (proportion of country-years in events)
        - mean_severity: Mean annual loss when losses occur (% below event start baseline)
        - expected_loss: Expected annual loss per country-year
        - total_exposure: Total country-year observations
        - event_exposure: Number of country-years in events
    """
    
    # Total exposure (all country-years)
    total_exposure = len(df_events)
    
    if total_exposure == 0:
        return {
            'frequency': 0.0,
            'mean_severity': 0.0,
            'expected_loss': 0.0,
            'total_exposure': 0,
            'event_exposure': 0
        }
    
    # Event exposure (country-years in events)
    event_years = df_events[df_events['in_event'] == 1].copy()
    event_exposure = len(event_years)
    
    # Annual prevalence frequency
    frequency = event_exposure / total_exposure
    
    if event_exposure == 0:
        return {
            'frequency': 0.0,
            'mean_severity': 0.0,
            'expected_loss': 0.0,
            'total_exposure': total_exposure,
            'event_exposure': 0
        }
    
    # Calculate annual losses relative to EVENT START baseline (not current year baseline)
    annual_losses = []
    
    # Group by country to find event start years
    for country in event_years['country'].unique():
        country_events = event_years[event_years['country'] == country].copy()
        
        # Find event boundaries for this country
        # Identify start of events (where in_event changes from 0 to 1 or is first observation)
        country_all = df_events[df_events['country'] == country].sort_values('year')
        country_all['event_start'] = (
            (country_all['in_event'] == 1) & 
            (country_all['in_event'].shift(1, fill_value=0) == 0)
        )
        
        # For each event year, find the corresponding event start baseline
        for _, row in country_events.iterrows():
            year = row['year']
            fx_index = row['fx_index']
            
            # Find the event start year for this observation
            # Look backwards to find when this event started
            country_subset = country_all[country_all['year'] <= year]
            event_starts = country_subset[country_subset['event_start'] == True]
            
            if len(event_starts) > 0:
                # Use the baseline from the most recent event start
                event_start_baseline = event_starts.iloc[-1]['baseline']
                
                # Calculate annual loss relative to event start baseline
                annual_loss = ((fx_index - event_start_baseline) / event_start_baseline * 100)
                
                # Only include losses (negative deviations)
                if annual_loss <= 0:
                    annual_losses.append(abs(annual_loss))
    
    if len(annual_losses) == 0:
        return {
            'frequency': 0.0,
            'mean_severity': 0.0,
            'expected_loss': 0.0,
            'total_exposure': total_exposure,
            'event_exposure': event_exposure
        }
    
    # Mean severity (absolute value of losses below event start baseline)
    mean_severity = np.mean(annual_losses)
    
    # Expected annual loss per country-year
    expected_loss = frequency * mean_severity
    
    return {
        'frequency': frequency,
        'mean_severity': mean_severity,
        'expected_loss': expected_loss,
        'total_exposure': total_exposure,
        'event_exposure': event_exposure
    }

def calculate_annual_risk_premiums(
    
    pooled_metrics,
    confidence_level=0.95,
    expense_ratio=0.15,
    profit_margin=0.10,
    capital_charge=0.15,
    diversification_credit=0.0
):
    """
    Calculate annual risk-based insurance premiums using prevalence and annual loss approach.
    
    This function calculates premiums based on annual prevalence (proportion of country-years
    experiencing losses) and the distribution of annual losses when they occur. This approach
    is more appropriate for annual capital adequacy requirements.
    
    Parameters
    ----------
    pooled_metrics : dict
        Output from frequency_and_severity_pooled function with keys:
        - frequency: Annual prevalence rate (proportion of country-years with losses)
        - mean_severity: Mean annual loss when losses occur (%)
        - std_severity: Standard deviation of annual losses
        - p95_severity: 95th percentile of annual losses
        - p99_severity: 99th percentile of annual losses
        - expected_loss: Expected annual loss per country-year
    confidence_level : float, default 0.95
        Confidence level for annual capital requirements (0.95 or 0.99)
    expense_ratio : float, default 0.15
        Administrative expense ratio as fraction of gross premium
    profit_margin : float, default 0.10
        Target profit margin as fraction of gross premium  
    capital_charge : float, default 0.15
        Annual capital charge rate for unexpected losses
    diversification_credit : float, default 0.0
        Portfolio diversification credit (0.0 to 0.3) to reduce capital requirements
        
    Returns
    -------
    dict
        Dictionary with annual premium calculations and risk metrics
    """
    
    if pooled_metrics['expected_loss'] == 0:
        return {"error": "No expected annual loss - cannot calculate premiums"}
    
    # Extract annual prevalence metrics
    prevalence = pooled_metrics['frequency']  # Proportion of country-years with losses
    mean_annual_loss = pooled_metrics['mean_severity']  # Mean loss when it occurs
    expected_annual_loss = pooled_metrics['expected_loss']  # Expected loss per country-year
    
    # Validate inputs
    if np.isinf(prevalence) or np.isinf(expected_annual_loss):
        return {"error": "Invalid prevalence or expected loss - check data quality"}
    
    if prevalence == 0:
        return {"error": "Zero prevalence - no losses detected in sample period"}
    
    # Determine capital loss level based on confidence level
    if confidence_level == 0.95:
        capital_annual_loss = pooled_metrics['p95_severity']
    elif confidence_level == 0.99:
        capital_annual_loss = pooled_metrics['p99_severity']
    else:
        # Normal approximation for other confidence levels
        from scipy.stats import norm
        z_score = norm.ppf(confidence_level)
        capital_annual_loss = mean_annual_loss + z_score * pooled_metrics['std_severity']
    
    # Apply diversification credit to reduce capital requirements
    diversified_capital_loss = capital_annual_loss * (1 - diversification_credit)
    
    # Annual premium calculation components
    # 1. Pure premium = expected annual loss per country-year
    pure_annual_premium = expected_annual_loss
    
    # 2. Risk margin for unexpected annual losses
    # Based on the difference between capital loss and expected loss at the prevalence rate
    unexpected_annual_loss = prevalence * diversified_capital_loss - expected_annual_loss
    annual_risk_margin = max(unexpected_annual_loss * capital_charge, 0)
    
    # 3. Technical premium (pure + risk margin)
    technical_annual_premium = pure_annual_premium + annual_risk_margin
    
    # 4. Gross premium with expense and profit loadings
    loading_denominator = 1 - expense_ratio - profit_margin
    if loading_denominator <= 0:
        raise ValueError("Combined expense ratio and profit margin must be < 100%")
    
    gross_annual_premium = technical_annual_premium / loading_denominator
    
    # Calculate risk and capital metrics
    premium_multiple = gross_annual_premium / pure_annual_premium if pure_annual_premium > 0 else np.inf
    capital_loss_multiple = diversified_capital_loss / mean_annual_loss if mean_annual_loss > 0 else np.inf
    risk_margin_ratio = annual_risk_margin / pure_annual_premium if pure_annual_premium > 0 else 0
    
    # Calculate implied annual probability of capital loss being exceeded
    capital_exceedance_prob = 1 - confidence_level
    
    return {
        # Input parameters
        'confidence_level': confidence_level,
        'expense_ratio': expense_ratio,
        'profit_margin': profit_margin,
        'capital_charge': capital_charge,
        'diversification_credit': diversification_credit,
        
        # Annual prevalence risk metrics  
        'annual_prevalence': prevalence,
        'mean_annual_loss': mean_annual_loss,
        'capital_annual_loss': capital_annual_loss,
        'diversified_capital_loss': diversified_capital_loss,
        'expected_annual_loss': expected_annual_loss,
        
        # Annual premium components
        'pure_annual_premium': pure_annual_premium,
        'annual_risk_margin': annual_risk_margin,
        'technical_annual_premium': technical_annual_premium,
        'gross_annual_premium': gross_annual_premium,
        
        # Risk and capital ratios
        'premium_multiple': premium_multiple,
        'capital_loss_multiple': capital_loss_multiple,
        'risk_margin_ratio': risk_margin_ratio,
        'capital_exceedance_prob': capital_exceedance_prob,
        
        # Legacy naming for backward compatibility
        'frequency': prevalence,
        'mean_severity': mean_annual_loss,
        'capital_severity': diversified_capital_loss,
        'pure_premium': pure_annual_premium,
        'risk_margin': annual_risk_margin,
        'technical_premium': technical_annual_premium,
        'gross_premium': gross_annual_premium,
        'capital_coverage_ratio': capital_loss_multiple
    }

def calculate_simple_risk_premium(df_events):
    """
    Calculate prevalence, annual severity, and pure risk premium from df_events.
    
    Parameters
    ----------
    df_events : pd.DataFrame
        DataFrame from identify_events() with columns:
        - country: 3-letter ISO country code
        - year: Year as integer
        - in_event: Binary indicator (1 if in event, 0 otherwise)
        - annual_loss: Annual loss as fraction (negative values)
        
    Returns
    -------
    dict
        Dictionary with:
        - prevalence: Proportion of country-years in events
        - mean_annual_severity: Mean annual loss when losses occur (%)
        - pure_risk_premium: Expected annual loss per country-year (%)
        - total_exposure: Total country-year observations
        - event_exposure: Number of country-years in events
    """
    
    # Total exposure (all country-years)
    total_exposure = len(df_events)
    
    if total_exposure == 0:
        return {
            'prevalence': 0.0,
            'mean_annual_severity': 0.0,
            'pure_risk_premium': 0.0,
            'total_exposure': 0,
            'event_exposure': 0
        }
    
    # Event exposure (country-years in events)
    event_years = df_events[df_events['in_event'] == 1]
    event_exposure = len(event_years)
    
    # Prevalence rate
    prevalence = event_exposure / total_exposure
    
    if event_exposure == 0:
        return {
            'prevalence': 0.0,
            'mean_annual_severity': 0.0,
            'pure_risk_premium': 0.0,
            'total_exposure': total_exposure,
            'event_exposure': 0
        }
    
    # Annual severity (convert to percentage and take absolute value)
    annual_losses = event_years['annual_loss'].abs() * 100
    mean_annual_severity = annual_losses.mean()
    
    # Pure risk premium (expected annual loss per country-year)
    pure_risk_premium = prevalence * mean_annual_severity
    
    return {
        'prevalence': prevalence,
        'mean_annual_severity': mean_annual_severity,
        'pure_risk_premium': pure_risk_premium,
        'total_exposure': total_exposure,
        'event_exposure': event_exposure
    }
def calculate_event_based_risk_premium(events, df_events):
    """
    Calculate event-based frequency, severity, and pure risk premium.
    
    Parameters
    ----------
    events : pd.DataFrame
        Event-level data from identify_events() with columns:
        - Country: 3-letter ISO country code
        - Start_Date, End_Date: Event dates
        - Duration: Event length in years
        - Drawdown_Area: Total event loss (%)
    df_events : pd.DataFrame
        Country-year data with columns:
        - country: 3-letter ISO country code
        - year: Year as integer
        
    Returns
    -------
    dict
        Dictionary with:
        - frequency: Events per country-year at risk (time-at-risk adjusted)
        - mean_severity: Mean total event loss when events occur (%)
        - pure_risk_premium: Expected event loss per country-year at risk (%)
        - total_exposure: Total country-years in dataset
        - time_at_risk: Country-years not in events (exposure for new events)
        - total_events: Number of events detected
    """
    
    # Total exposure (all country-years in dataset)
    total_exposure = len(df_events)
    
    if total_exposure == 0 or len(events) == 0:
        return {
            'frequency': 0.0,
            'mean_severity': 0.0,
            'pure_risk_premium': 0.0,
            'total_exposure': total_exposure,
            'time_at_risk': total_exposure,
            'total_events': 0
        }
    
    # Calculate time at risk (exclude country-years already in events)
    total_event_duration = events['Duration'].sum()
    time_at_risk = total_exposure - total_event_duration
    
    # Number of events
    total_events = len(events)
    
    # Event-based frequency (events per country-year at risk)
    if time_at_risk > 0:
        frequency = total_events / time_at_risk
    else:
        frequency = 0.0
    
    # Event-based severity (mean total event loss)
    mean_severity = events['Drawdown_Area'].mean()
    
    # Pure risk premium (expected event loss per country-year at risk)
    pure_risk_premium = frequency * mean_severity
    
    return {
        'frequency': frequency,
        'mean_severity': mean_severity,
        'pure_risk_premium': pure_risk_premium,
        'total_exposure': total_exposure,
        'time_at_risk': time_at_risk,
        'total_events': total_events
    }




In [ ]:
hybrid = build_hybrid_index_and_ma_df(start="1994-01-01")
events, df_events = identify_events(hybrid, trigger_threshold=0.1, end_threshold=0.1)
# df_events["annual_loss"].sum(), events["Drawdown_Area"].sum()/100
# Usage example:
results = calculate_simple_risk_premium(df_events)

print(f"Portfolio prevalence: {results['prevalence']:.4f} ({results['prevalence']:.1%})")
print(f"Mean annual severity when losses occur: {results['mean_annual_severity']:.2f}%")
print(f"Pure risk premium: {results['pure_risk_premium']:.4f} ({results['pure_risk_premium']:.2f}%)")
print(f"Exposure: {results['event_exposure']} event years out of {results['total_exposure']} total")

event_results = calculate_event_based_risk_premium(events, df_events)

print(f"Event-based frequency: {event_results['frequency']:.6f} events per country-year at risk")
print(f"Mean event severity: {event_results['mean_severity']:.2f}% (total event loss)")
print(f"Pure risk premium: {event_results['pure_risk_premium']:.4f} ({event_results['pure_risk_premium']:.2f}%)")
print(f"Time at risk: {event_results['time_at_risk']} country-years out of {event_results['total_exposure']} total")
print(f"Total events: {event_results['total_events']}")

In [ ]:
907-86+590

In [ ]:
# Add this diagnostic code after your identify_events call:
print("=== DIAGNOSTIC: Breakdown of the Difference ===")

# 1. Check if they use the same baseline reference
annual_loss_total = df_events["annual_loss"].sum() * 100  # Convert to percentage
drawdown_total = events["Drawdown_Area"].sum()

print(f"Annual loss total (as %): {annual_loss_total:.1f}")
print(f"Drawdown Area total: {drawdown_total:.1f}")
print(f"Ratio: {annual_loss_total / drawdown_total:.3f}")

# 2. Check number of country-years in each calculation
event_rows_annual = len(df_events[df_events["annual_loss"] != 0])
total_event_duration = events["Duration"].sum()

print(f"Country-years with annual_loss: {event_rows_annual}")
print(f"Total event duration (should match): {total_event_duration}")

# 3. Sample a specific event to trace the calculation
if len(events) > 0:
    sample_event = events.iloc[50]
    print(f"\nSample event: {sample_event['Country']} {sample_event['Start_Date']}-{sample_event['End_Date']}")
    print(f"Event Drawdown_Area: {sample_event['Drawdown_Area']:.2f}")
    
    # Find corresponding annual losses
    event_mask = (
        (df_events['country'] == sample_event['Country']) &
        (df_events['year'] >= sample_event['Start_Date']) &
        (df_events['year'] <= sample_event['End_Date'])
    )
    event_annual_losses = df_events[event_mask]['annual_loss'].sum() * 100
    print(f"Corresponding annual losses: {event_annual_losses:.2f}")
    print(f"Ratio for this event: {event_annual_losses / sample_event['Drawdown_Area']:.3f}")

In [ ]:
events[events["Country"] == "MLI"]

In [ ]:
x = df_events[df_events["country"] == "MLI"].reset_index()
x

In [ ]:
np.sum(x[20:24]["annual_loss"])


In [ ]:
import pymc as pm
import arviz as az

# Build panel of all country-years
panel = hybrid.rename(columns={"country": "Country", "year": "Year"})
panel["event"] = 0

# Mark event years
for _, row in events.iterrows():
    mask = (
        (panel["Country"] == row["Country"]) &
        (panel["Year"] >= row["Start_Date"]) &
        (panel["Year"] <= row["End_Date"])
    )
    panel.loc[mask, "event"] = 1

# Encode categorical variables
panel["year_idx"], year_index = pd.factorize(panel["Year"])
panel["country_idx"], country_index = pd.factorize(panel["Country"])

y = panel["event"].values
n_years = len(year_index)
n_countries = len(country_index)

# -----
# Bayesian GLMM in PyMC
# -----
with pm.Model() as model:
    # Priors
    mu_alpha = pm.Normal("mu_alpha", mu=0, sigma=2)         # global intercept
    sigma_year = pm.HalfNormal("sigma_year", sigma=1)       # variance of year effects
    sigma_country = pm.HalfNormal("sigma_country", sigma=1) # optional country random effect

    # Random intercepts
    year_offset = pm.Normal("year_offset", mu=0, sigma=1, shape=n_years)
    country_offset = pm.Normal("country_offset", mu=0, sigma=1, shape=n_countries)

    year_effect = pm.Deterministic("year_effect", year_offset * sigma_year)
    country_effect = pm.Deterministic("country_effect", country_offset * sigma_country)

    # Linear predictor
    eta = (
        mu_alpha
        + year_effect[panel["year_idx"].values]
        + country_effect[panel["country_idx"].values]
    )

    # Likelihood
    p = pm.math.sigmoid(eta)
    y_obs = pm.Bernoulli("y_obs", p=p, observed=y)

    # Inference
    trace = pm.sample(1000, tune=1000, chains=2, target_accept=0.9, random_seed=42)

# Summarize posterior
az.summary(trace, var_names=["mu_alpha","sigma_year","sigma_country"])

In [ ]:
# Build panel of all country-years
panel = hybrid.rename(columns={"country": "Country", "year": "Year"})
panel["event"] = 0

# Mark event years
for _, row in events.iterrows():
    mask = (
        (panel["Country"] == row["Country"]) &
        (panel["Year"] >= row["Start_Date"]) &
        (panel["Year"] <= row["End_Date"])
    )
    panel.loc[mask, "event"] = 1
panel

In [ ]:
import pymc as pm
import arviz as az
import numpy as np

# Assume `panel` dataframe exists with columns: country, year, event (0/1)
# Encode country and year as integers
panel["country_id"] = panel["Country"].astype("category").cat.codes
panel["year_id"] = panel["Year"].astype("category").cat.codes

n_countries = panel["country_id"].nunique()
n_years = panel["year_id"].nunique()

with pm.Model() as fast_model:
    # Global intercept
    alpha = pm.Normal("alpha", mu=0, sigma=2)
    
    # Year random effects (systemic/global shocks)
    sigma_year = pm.HalfNormal("sigma_year", sigma=1)
    year_offset = pm.Normal("year_offset", mu=0, sigma=1, shape=n_years)
    year_effect = pm.Deterministic("year_effect", year_offset * sigma_year)
    
    # Country random effects
    sigma_country = pm.HalfNormal("sigma_country", sigma=1)
    country_offset = pm.Normal("country_offset", mu=0, sigma=1, shape=n_countries)
    country_effect = pm.Deterministic("country_effect", country_offset * sigma_country)
    
    # Linear predictor
    logit_p = (
        alpha
        + year_effect[panel["year_id"].values]
        + country_effect[panel["country_id"].values]
    )
    
    # Likelihood
    event = pm.Bernoulli("event", logit_p=logit_p, observed=panel["event"].values)

    # Variational inference (ADVI)
    approx = pm.fit(method="advi", n=50000)
    trace = approx.sample(2000)

# Summarize key results
summary = az.summary(trace, var_names=["alpha", "sigma_year", "sigma_country"])
print(summary)


In [ ]:
# Streamlined analysis workflow with annual prevalence approach
hybrid = build_hybrid_index_and_ma_df(start="1994-01-01")
events = identify_events(hybrid, trigger_threshold=0.1, end_threshold=0.1)

print("=== EVENT DETECTION SUMMARY ===")
print(f"Total events detected: {len(events)}")
print(f"Average event duration: {events['Duration'].mean():.1f} years")
print(f"Total event-years: {events['Duration'].sum():.0f}")

# Create annual event data for diagnostic
annual_data = create_annual_event_data(events, hybrid)
total_country_years = len(annual_data)
event_country_years = len(annual_data[annual_data['In_Event'] == 1])

print(f"\n=== ANNUAL PREVALENCE DATA ===")
print(f"Total country-year observations: {total_country_years}")
print(f"Country-years in events: {event_country_years}")
print(f"Annual prevalence rate: {event_country_years/total_country_years:.4f}")

# Calculate pooled metrics using annual prevalence
pooled_metrics = frequency_and_severity_pooled(events, hybrid)
print(f"\n=== POOLED PORTFOLIO METRICS (Annual Prevalence) ===")
for key, value in pooled_metrics.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")

# Calculate country-level metrics
country_metrics = frequency_and_severity_by_country(events, hybrid)
print(f"\n=== TOP 10 COUNTRIES BY EXPECTED ANNUAL LOSS ===")
top_countries = country_metrics[country_metrics['Expected_Loss'] > 0].nlargest(10, 'Expected_Loss')
print(top_countries.round(4))

# Calculate premiums using annual prevalence approach
premium_results = calculate_annual_risk_premiums(
    pooled_metrics, 
    confidence_level=0.95,
    diversification_credit=0.1  # 10% portfolio diversification credit
)

if "error" not in premium_results:
    print(f"\n=== ANNUAL RISK PREMIUM CALCULATION ===")
    print(f"Annual prevalence rate: {premium_results['annual_prevalence']:.1%}")
    print(f"Mean annual loss when it occurs: {premium_results['mean_annual_loss']:.1f}%")
    print(f"95th percentile annual loss: {premium_results['capital_annual_loss']:.1f}%")
    print(f"Expected annual loss per country-year: {premium_results['expected_annual_loss']:.4f}")
    
    print(f"\n--- Premium Components ---")
    print(f"Pure annual premium: {premium_results['pure_annual_premium']:.4f} ({premium_results['pure_annual_premium']*100:.2f}%)")
    print(f"Annual risk margin: {premium_results['annual_risk_margin']:.4f} ({premium_results['risk_margin_ratio']:.1%} of pure premium)")
    print(f"Technical annual premium: {premium_results['technical_annual_premium']:.4f}")
    print(f"Gross annual premium: {premium_results['gross_annual_premium']:.4f} ({premium_results['gross_annual_premium']*100:.2f}%)")
    
    print(f"\n--- Risk Metrics ---")
    print(f"Premium multiple: {premium_results['premium_multiple']:.2f}x")
    print(f"Capital loss multiple: {premium_results['capital_loss_multiple']:.2f}x")
    print(f"Capital exceedance probability: {premium_results['capital_exceedance_prob']:.1%}")
    
    print(f"\nFinal annual premium rate: {premium_results['gross_annual_premium']:.4f} ({premium_results['gross_annual_premium']*100:.2f}%)")
else:
    print(f"Premium calculation error: {premium_results['error']}")

# Show sample of annual event data for transparency
print(f"\n=== SAMPLE ANNUAL EVENT DATA ===")
sample_events = annual_data[annual_data['In_Event'] == 1].head(10)
print(sample_events[['Country', 'Year', 'Annual_Loss', 'Event_Year', 'Total_Event_Duration']].round(4))

In [ ]:
# COMPARISON: Old vs New Approach
print("=== COMPARISON: Event-Level vs Annual Prevalence Approach ===")

# For comparison, let's calculate the old way (event-level) manually
if len(events) > 0:
    # Old approach: frequency = events per country per year (time-at-risk corrected)
    total_calendar_years = hybrid.groupby('country')['year'].nunique().sum()
    total_event_duration = events['Duration'].sum()
    total_time_at_risk = total_calendar_years - total_event_duration
    old_frequency = len(events) / total_time_at_risk if total_time_at_risk > 0 else 0
    old_mean_severity = events['Drawdown_Area'].mean()  # Total event losses
    old_expected_loss = old_frequency * old_mean_severity
    
    print(f"\nOLD APPROACH (Event-Level):")
    print(f"  Events per country-year (time-at-risk corrected): {old_frequency:.6f}")
    print(f"  Mean severity (total event loss): {old_mean_severity:.2f}")
    print(f"  Expected loss: {old_expected_loss:.6f}")
    
    print(f"\nNEW APPROACH (Annual Prevalence):")
    print(f"  Annual prevalence rate: {pooled_metrics['frequency']:.6f}")
    print(f"  Mean annual severity: {pooled_metrics['mean_severity']:.2f}")
    print(f"  Expected annual loss: {pooled_metrics['expected_loss']:.6f}")
    
    print(f"\nKEY DIFFERENCES:")
    print(f"  Frequency ratio (new/old): {pooled_metrics['frequency']/old_frequency:.2f}x")
    print(f"  Severity ratio (new/old): {pooled_metrics['mean_severity']/old_mean_severity:.2f}x")
    print(f"  Expected loss ratio (new/old): {pooled_metrics['expected_loss']/old_expected_loss:.2f}x")
    
    print(f"\nINTERPRETATION:")
    print(f"  - Annual prevalence approach shows {pooled_metrics['frequency']:.1%} of country-years experience losses")
    print(f"  - Average annual loss when it occurs: {pooled_metrics['mean_severity']:.1f}% below baseline")
    print(f"  - This approach is better for annual capital requirements since it captures")
    print(f"    the yearly distribution of losses rather than total event losses")
else:
    print("No events detected for comparison")

In [ ]:
# ENHANCED ANNUAL PREVALENCE PREMIUM ANALYSIS
print("=== SENSITIVITY ANALYSIS: Annual Prevalence Premium Approach ===")

# Test different confidence levels and diversification credits
confidence_levels = [0.95, 0.99]
diversification_credits = [0.0, 0.1, 0.2]

premium_scenarios = []

for conf_level in confidence_levels:
    for div_credit in diversification_credits:
        scenario_results = calculate_annual_risk_premiums(
            pooled_metrics,
            confidence_level=conf_level,
            diversification_credit=div_credit,
            expense_ratio=0.15,
            profit_margin=0.10,
            capital_charge=0.15
        )
        
        if "error" not in scenario_results:
            premium_scenarios.append({
                'Confidence_Level': f"{conf_level:.0%}",
                'Diversification_Credit': f"{div_credit:.0%}",
                'Capital_Annual_Loss': f"{scenario_results['capital_annual_loss']:.1f}%",
                'Diversified_Capital_Loss': f"{scenario_results['diversified_capital_loss']:.1f}%",
                'Risk_Margin': f"{scenario_results['annual_risk_margin']:.4f}",
                'Gross_Premium': f"{scenario_results['gross_annual_premium']:.4f}",
                'Premium_Multiple': f"{scenario_results['premium_multiple']:.2f}x"
            })

premium_df = pd.DataFrame(premium_scenarios)
print("\nPremium Sensitivity Analysis:")
print(premium_df.to_string(index=False))

print(f"\n=== KEY INSIGHTS FROM ANNUAL PREVALENCE APPROACH ===")
print(f"• Portfolio prevalence: {pooled_metrics['frequency']:.1%} of country-years experience currency losses")
print(f"• Typical annual loss: {pooled_metrics['mean_severity']:.1f}% below baseline when losses occur")
print(f"• Severe annual loss (95th percentile): {pooled_metrics['p95_severity']:.1f}% below baseline")
print(f"• Expected annual loss: {pooled_metrics['expected_loss']:.4f} per country-year")

print(f"\n=== ADVANTAGES OF ANNUAL PREVALENCE MODEL ===")
print(f"• Capital requirements based on annual loss distribution (not multi-year event totals)")
print(f"• Prevalence rate shows proportion of exposure years with losses")
print(f"• Annual severity reflects yearly capital needs for solvency")
print(f"• Diversification credit can be applied to reduce capital requirements")
print(f"• More intuitive for insurance capital adequacy frameworks")

In [ ]:
from scipy.stats import lognorm, norm
from tqdm import tqdm
import matplotlib.pyplot as plt

poisson_summary = poisson_with_offset(events, hdf)
lognormal_summary = lognormal_estimattion(events)
# print(poisson_summary)
# print(lognormal_summary)

b0 = -2.7861
se_b0 = 0.107
mu_log = lognormal_summary['mu']
sigma_log = lognormal_summary['sigma']

# Simulation settings
N_SIMS = 20000
SEED = 42

# Pricing settings (example)
expense_ratio = 0.10    # 10% expenses
profit_margin = 0.05    # 5% profit
solvency_q = 0.995      # for TVaR/VaR (e.g., 99.5%)

# Optional policy terms (set None to disable)
POLICY_LIMIT = None     # e.g., 1e4
DEDUCTIBLE = 0.0        # per-claim deductible
QUOTA_SHARE = 1.0       # insurer retains this fraction (0-1)


# Standardize column names if necessary
# events: expect 'Country' and 'Drawdown_Area'
# hybrid: expect 'country' and 'year'
if "Country" not in events.columns and "country" in events.columns:
    events.rename(columns={"country": "Country"}, inplace=True)
if "Drawdown_Area" not in events.columns:
    raise ValueError("events CSV must contain 'Drawdown_Area' column")

exposure = hybrid.groupby("country")["year"].nunique().reset_index().rename(columns={"country":"Country","year":"exposure_years"})
counts = events.groupby("Country").size().reset_index(name="event_count")
df = pd.merge(exposure, counts, on="Country", how="outer").fillna(0)
df["event_count"] = df["event_count"].astype(int)

# analytic lambda per year
lambda_year = np.exp(b0)

# analytic expected events & expected loss (using analytic mean severity)
sev_dist = lognorm(s=sigma_log, scale=np.exp(mu_log))
mean_severity = sev_dist.mean()
var_severity = sev_dist.var()

df["expected_events"] = df["exposure_years"] * lambda_year
df["analytic_expected_loss"] = df["expected_events"] * mean_severity

print("Overall frequency lambda (events / country / year):", lambda_year)
print("Lognormal severity: mu_log =", mu_log, "sigma_log =", sigma_log)
print("Mean severity (analytic):", mean_severity)
print("Number of countries considered:", df.shape[0])

# ---------- 2. Monte Carlo simulation of aggregate losses ----------
rng = np.random.default_rng(SEED)

def sample_severity_lognormal(n, mu_log, sigma_log, rng=None):
    if n <= 0:
        return np.array([])
    # lognormal parameterization: s = sigma, scale = exp(mu)
    return lognorm(s=sigma_log, scale=np.exp(mu_log)).rvs(size=n, random_state=rng)

def simulate_portfolio(df, b0, mu_log, sigma_log, n_sims=20000,
                       policy_limit=None, deductible=0.0, quota_share=1.0, seed=42):
    rng = np.random.default_rng(seed)
    portfolio_losses = np.zeros(n_sims)
    per_country_losses = {c: np.zeros(n_sims) for c in df["Country"].values}

    for _, row in df.iterrows():
        country = row["Country"]
        T = int(row["exposure_years"])
        lam = np.exp(b0) * T
        # simulate counts
        Nsim = rng.poisson(lam, size=n_sims)
        losses = np.zeros(n_sims)
        pos_idx = np.where(Nsim > 0)[0]
        for i in pos_idx:
            n_claims = int(Nsim[i])
            sev = sample_severity_lognormal(n_claims, mu_log, sigma_log, rng=rng)
            if deductible > 0:
                sev = np.maximum(sev - deductible, 0.0)
            if policy_limit is not None:
                sev = np.minimum(sev, policy_limit)
            total = sev.sum() if n_claims>0 else 0.0
            retained = total * quota_share
            losses[i] = retained
        per_country_losses[country] = losses
        portfolio_losses += losses

    return per_country_losses, portfolio_losses

per_country_losses, portfolio_losses = simulate_portfolio(
    df, b0, mu_log, sigma_log,
    n_sims=N_SIMS, policy_limit=POLICY_LIMIT,
    deductible=DEDUCTIBLE, quota_share=QUOTA_SHARE, seed=SEED
)

# ---------- 3. Summarize results ----------
def portfolio_metrics(portfolio_losses, qs=[0.90,0.95,0.99,0.995]):
    out = {}
    out["mean"] = float(np.mean(portfolio_losses))
    out["std"] = float(np.std(portfolio_losses, ddof=1))
    for q in qs:
        v = float(np.quantile(portfolio_losses, q))
        out[f"VaR_{int(q*100)}"] = v
        out[f"TVaR_{int(q*100)}"] = float(portfolio_losses[portfolio_losses >= v].mean()) if np.sum(portfolio_losses >= v)>0 else v
    return out

p_metrics = portfolio_metrics(portfolio_losses, qs=[0.90,0.95,0.99,solvency_q])
print("\nPortfolio metrics (from simulation):")
for k,v in p_metrics.items():
    print(f"  {k}: {v}")

# Per-country summary (simulated)
country_rows = []
for c, sims in per_country_losses.items():
    country_rows.append({
        "Country": c,
        "ExpLoss_sim": float(np.mean(sims)),
        "StdLoss_sim": float(np.std(sims, ddof=1)),
        "MedianLoss_sim": float(np.median(sims)),
        "AnalyticExpLoss": float(df.loc[df["Country"]==c, "analytic_expected_loss"].values[0])
    })
country_df = pd.DataFrame(country_rows).set_index("Country")
print("\nSample country summary:")
print(country_df.sort_values("AnalyticExpLoss", ascending=False).head(8))

# ---------- 4. Simple premium calculation ----------
pure_premium = p_metrics["mean"]
risk_margin = p_metrics[f"TVaR_{int(solvency_q*100)}"] - pure_premium  # TVaR - mean
suggested_premium = pure_premium * (1 + expense_ratio + profit_margin) + risk_margin

print("\nPremium calculation (portfolio-level):")
print("  Pure premium (expected loss):", pure_premium)
print(f"  Expense ratio: {expense_ratio*100:.1f}%, profit margin: {profit_margin*100:.1f}%")
print("  Risk margin (TVaR - mean):", risk_margin)
print("  Suggested premium (pure + loadings + risk margin):", suggested_premium)

# Allocate premium to countries by analytic expected loss share
total_analytic_loss = df["analytic_expected_loss"].sum()
if total_analytic_loss > 0:
    df["allocated_premium"] = df["analytic_expected_loss"] / total_analytic_loss * suggested_premium
else:
    df["allocated_premium"] = suggested_premium / df.shape[0]

# Save outputs
df_out = df[["Country","exposure_years","event_count","expected_events","analytic_expected_loss","allocated_premium"]]
df_out.to_csv("country_premium_allocations.csv", index=False)
print("\nSaved per-country allocations to country_premium_allocations.csv")

# ---------- 5. Plots ----------
plt.figure(figsize=(8,5))
plt.hist(portfolio_losses, bins=100, density=False, alpha=0.6)
plt.axvline(p_metrics["mean"], color="black", linestyle="--", label="Mean")
plt.axvline(p_metrics["VaR_99"], color="red", linestyle="-.", label="VaR99")
plt.axvline(p_metrics[f"TVaR_{int(solvency_q*100)}"], color="purple", linestyle=":", label=f"TVaR{int(solvency_q*100)}")
plt.title("Portfolio aggregate loss distribution (simulated)")
plt.xlabel("Aggregate loss (portfolio)")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.savefig("portfolio_loss_hist.png", dpi=150)
print("Saved portfolio loss histogram to portfolio_loss_hist.png")

# ---------- 6. Optional: Parametric bootstrap for parameter uncertainty ----------
def bootstrap_parameter_uncertainty(df, severity_array, b0, se_b0, n_boot=200, sims_per_boot=2000, seed=1):
    rng = np.random.default_rng(seed)
    boot_means = np.zeros(n_boot)
    for i in tqdm(range(n_boot), desc="Parametric bootstrap"):
        # sample b0*
        b0_star = rng.normal(b0, se_b0)
        # nonparametric bootstrap of severity data and refit lognormal
        resample = rng.choice(severity_array, size=len(severity_array), replace=True)
        try:
            shape_star, loc_star, scale_star = lognorm.fit(resample, floc=0)
            sigma_star = shape_star
            mu_star = np.log(scale_star)
        except Exception:
            sigma_star = sigma_log
            mu_star = mu_log
        _, portfolio_losses_star = simulate_portfolio(df, b0_star, mu_star, sigma_star,
                                                      n_sims=sims_per_boot, policy_limit=POLICY_LIMIT,
                                                      deductible=DEDUCTIBLE, quota_share=QUOTA_SHARE, seed=rng.integers(1e9))
        boot_means[i] = portfolio_losses_star.mean()
    return boot_means

# To run the bootstrap uncomment the following lines (may be slow):
# severity_array = events["Drawdown_Area"].dropna().values
# boot_means = bootstrap_parameter_uncertainty(df, severity_array, b0, se_b0, n_boot=200, sims_per_boot=2000, seed=123)
# print("Bootstrap portfolio mean mean/std:", boot_means.mean(), boot_means.std())

# ---------- End ----------

In [ ]:
hdf = build_hybrid_index_and_ma_df(start= "2017-01-01")
df = hdf.dasdfasdfdaf
trigger_threshold = 0.1
end_threshold = 0.0
results = []
country = "AGO"
country_data = df[df.country == country]
# for loop code

year         = country_data["year"].values
fx_index     = country_data["fx_index"].values
baseline     = country_data["baseline"].values

peak_indices = argrelextrema(fx_index, np.greater)[0]
if fx_index[-1] > fx_index[-2]:
    peak_indices = np.append(peak_indices, len(fx_index) - 1)

for t_idx in np.arange(len(year)):
    possible_event_peaks = peak_indices[peak_indices > t_idx]
    if possible_event_peaks.size == 0:
        continue

    for p_idx in possible_event_peaks:
        trigger_level = baseline[t_idx] * (1. + trigger_threshold)  # Use trigger threshold
        end_level     = baseline[t_idx] * (1. + end_threshold)  # This changes with end_threshold!
        if fx_index[p_idx] >= trigger_level:
            end_date = None
            event_duration = None
            max_peak = None
            max_devaluation = None
            drawdown_area = None
            
            intermediate_troughs = 0
            for i in range(t_idx+1, p_idx):
                if fx_index[i] < trigger_level:  # Check against trigger level
                    intermediate_troughs += 1
            if intermediate_troughs > 0:
                continue # skip this event if there are intermediate troughs, as it is not a clean rise and fall
            print("trough",t_idx)
            print("possible peaks", possible_event_peaks)
            print("peak", p_idx)
            for idx in range(p_idx+1, len(fx_index)):
                if fx_index[idx] <= end_level:
                    print("end", idx)
                    print("end level delta", end_level- fx_index[idx])
                    # time measures
                    end_date = year[idx]
                    event_duration = year[idx] - year[t_idx]

                    # max amplitude
                    max_peak = p_idx
                    for i in peak_indices[(peak_indices>p_idx) & (peak_indices<idx)]:
                        if fx_index[i] > fx_index[max_peak]:
                            max_peak = i
                    max_devaluation = (fx_index[max_peak] - baseline[t_idx]) / baseline[t_idx] * 100

                    # total dollar loss in precent of CFADS
                    # Replace fx_index values at start and end with baseline in a copy
                    baseline_to_baseline        = np.array(fx_index, copy = True)
                    baseline_to_baseline[t_idx] = np.maximum(baseline_to_baseline[t_idx], baseline[t_idx])
                    baseline_to_baseline[idx]   = np.maximum(baseline_to_baseline[idx],  baseline[t_idx])

                    integrand = np.maximum(baseline_to_baseline/baseline[t_idx] - 1, 0.0) * 100 # percentage deviation from baseline at event start date
                    drawdown_area = np.sum(integrand[t_idx:idx]) #sum of integrand over event period
                    print(drawdown_area)
                    break

            else: # if return condition not met, repeat with end period
                idx = len(fx_index) -1
                end_date = year[idx]
                event_duration = year[idx] - year[t_idx]
                max_peak = p_idx
                for i in peak_indices[(peak_indices>p_idx)]:
                    if fx_index[i] > fx_index[max_peak]:
                        max_peak = i
                max_devaluation = (fx_index[max_peak] - baseline[t_idx]) / baseline[t_idx] * 100
                baseline_to_baseline        = np.array(fx_index, copy = True)
                baseline_to_baseline[t_idx] = np.maximum(baseline_to_baseline[t_idx], baseline[t_idx])
                baseline_to_baseline[idx]   = np.maximum(baseline_to_baseline[idx],  baseline[t_idx])

                integrand = np.maximum(baseline_to_baseline/baseline[t_idx] - 1, 0.0) * 100 # percentage deviation from baseline at event start date
                drawdown_area = np.sum(integrand[t_idx:idx+1]) #sum of integrand over event period

            results.append({
                "Country": country, 
                "Start_Date": year[t_idx],
                "End_Date": end_date,
                "Peak_Date": year[max_peak],
                "Duration": event_duration,
                "Peak_Devaluation": max_devaluation,
                "Devaluation_at_End": (fx_index[idx] - baseline[t_idx]) / baseline[t_idx] * 100,
                "Drawdown_Area": drawdown_area
            })

dout = pd.DataFrame(results)

# x = np.maximum((fx_index-baseline[12])/baseline[12],0)[12:16]*100
# np.sum(x)
dout

In [ ]:
country_data